# Gaze and Pose-Based Interaction

Work by Antonio Vila Leis and Enric Baixauli Casañ

In [2]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import scipy.io as sio
from torchvision import transforms
from torch.utils.data import DataLoader

In [3]:


class LP300W(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.img_paths = []
        self.mat_paths = []

        for folder in os.listdir(root):
            fpath = os.path.join(root, folder)
            if not os.path.isdir(fpath):
                continue

            for file in os.listdir(fpath):
                if file.endswith(".jpg"):
                    img_path = os.path.join(fpath, file)
                    mat_path = img_path.replace(".jpg", ".mat")

                    if os.path.exists(mat_path):
                        self.img_paths.append(img_path)
                        self.mat_paths.append(mat_path)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert("RGB")
        mat = sio.loadmat(self.mat_paths[idx])

        pose = torch.tensor(mat["Pose_Para"][0][:3], dtype=torch.float32)

        if self.transform:
            img = self.transform(img)

        return img, pose


In [8]:


dataset = LP300W(
    root="/kaggle/input/datasets/adimukherjee1/300wlp/300W_LP",
    transform=transforms.ToTensor()
)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

img, pose = next(iter(loader))
print(img.shape, pose.shape)


KeyboardInterrupt: 